# Stripping `<think>…</think>` with FSTs

Two FST constructions for removing reasoning traces from LLM output:
1. **Strip anywhere** — remove all `<think>…</think>` blocks
2. **Strip at beginning only** — only remove a leading block
3. **Decomposition** — what source strings are consistent with an output prefix?

In [ ]:
from transduction import FST, EPSILON
from transduction.viz import visualize_automaton

OPEN = '<think>'
CLOSE = '</think>'
ALPHA = set('abcdefghijklmnopqrstuvwxyz <>/')

def s2t(s): return tuple(s)       # string → char tuple
def t2s(t): return ''.join(t)     # char tuple → string

---
## 1. Strip `<think>…</think>` anywhere

The FST non-deterministically chooses whether each `<` starts a tag match
or is just copied through.  When a full `<think>…</think>` is present the
matching path succeeds (stripping the block) while the copying path also
succeeds (keeping it).  So we may get **multiple outputs** — one per subset
of blocks stripped.  The FST is *not* functional.

This is the simplest construction; a deterministic (functional) version
needs "flush chains" to output buffered characters on a failed partial match.

In [ ]:
def strip_anywhere(open_tag, close_tag, alphabet):
    """Non-deterministic FST: strip all <open_tag>…<close_tag> blocks."""
    fst = FST()
    fst.add_start('copy')
    fst.add_stop('copy')

    # Copy mode: always allow copying any character
    for c in alphabet:
        fst.add_arc('copy', c, c, 'copy')

    # Non-deterministically start matching open tag
    fst.add_arc('copy', open_tag[0], EPSILON, ('mo', 0))
    for i in range(len(open_tag) - 1):
        target = 'inside' if i + 1 == len(open_tag) - 1 else ('mo', i + 1)
        fst.add_arc(('mo', i), open_tag[i + 1], EPSILON, target)

    # Inside: consume everything; non-deterministically match close tag
    for c in alphabet:
        fst.add_arc('inside', c, EPSILON, 'inside')
    fst.add_arc('inside', close_tag[0], EPSILON, ('mc', 0))
    for j in range(len(close_tag) - 1):
        target = 'copy' if j + 1 == len(close_tag) - 1 else ('mc', j + 1)
        fst.add_arc(('mc', j), close_tag[j + 1], EPSILON, target)

    return fst

T1 = strip_anywhere(OPEN, CLOSE, ALPHA)
print(f'{len(T1.states)} states')
display(visualize_automaton(T1))

In [ ]:
tests = [
    'hello',
    '<think>reason</think>answer',
    'before<think>stuff</think>after',
    '<think>a</think>b<think>c</think>d',
]
for s in tests:
    outputs = sorted(set(t2s(o) for o in T1.transduce(s2t(s))), key=len)
    print(f'{s!r:50s} → {outputs}')

Each input with *n* strippable blocks yields up to 2ⁿ outputs (one per
subset of blocks stripped).  The **shortest** output is the one with all
blocks removed — usually what we want.

---
## 2. Strip only at the beginning

Same idea, but matching is only attempted from the **start** state.
Once we enter `copy` mode (either after stripping or on the first
non-`<` character), no further matching happens — later `<think>` blocks
are copied through verbatim.

In [ ]:
def strip_beginning(open_tag, close_tag, alphabet):
    """Non-deterministic FST: strip <open_tag>…<close_tag> only at the start."""
    fst = FST()
    fst.add_start('start')
    fst.add_stop('start')   # accept empty string
    fst.add_stop('copy')

    # From start: copy any char and enter copy-only mode
    for c in alphabet:
        fst.add_arc('start', c, c, 'copy')

    # Non-deterministically try matching the open tag at position 0
    fst.add_arc('start', open_tag[0], EPSILON, ('mo', 0))
    for i in range(len(open_tag) - 1):
        target = 'inside' if i + 1 == len(open_tag) - 1 else ('mo', i + 1)
        fst.add_arc(('mo', i), open_tag[i + 1], EPSILON, target)

    # Inside: consume everything; match close tag
    for c in alphabet:
        fst.add_arc('inside', c, EPSILON, 'inside')
    fst.add_arc('inside', close_tag[0], EPSILON, ('mc', 0))
    for j in range(len(close_tag) - 1):
        target = 'copy' if j + 1 == len(close_tag) - 1 else ('mc', j + 1)
        fst.add_arc(('mc', j), close_tag[j + 1], EPSILON, target)

    # Copy mode: just copy, no more matching
    for c in alphabet:
        fst.add_arc('copy', c, c, 'copy')

    return fst

T2 = strip_beginning(OPEN, CLOSE, ALPHA)
print(f'{len(T2.states)} states')
display(visualize_automaton(T2))

In [ ]:
tests2 = [
    'hello',
    '<think>reason</think>answer',
    'before<think>stuff</think>after',            # tag not at beginning → copied
    '<think>a</think>b<think>c</think>d',         # greedy vs non-greedy!
]
for s in tests2:
    outputs = sorted(set(t2s(o) for o in T2.transduce(s2t(s))), key=len)
    print(f'{s!r:50s} → {outputs}')

Note the last example: `<think>a</think>b<think>c</think>d` produces three
outputs. The `inside` state can consume through the first `</think>` and
exit at the *second* one — this is the greedy `.*` interpretation, giving
output `d`. The non-greedy path exits at the first `</think>` and copies
the rest, giving `b<think>c</think>d`. Non-determinism gives us both.

---
## 3. Decomposition

**Decomposition** answers: given an output prefix *y*, what source strings
could produce it?  The **precover** P(*y*) is the set of all such source
strings.  It factors into:

- **Quotient** Q(*y*): source prefixes that have produced *y* and can
  continue — i.e., Q(*y*) · Σ\* ⊆ P(*y*).
- **Remainder** R(*y*): complete source strings *x* where *f*(*x*) = *y*
  exactly, with no continuation possible.

This is the core operation for constrained decoding: `decompose_next()`
computes Q and R for every possible next output symbol, telling you
which symbols are valid continuations.

For tractable enumeration, we use a **toy model** — single-character
delimiters `(` and `)` instead of multi-character tags — same structure,
smaller alphabet.

In [ ]:
from transduction.precover import Precover

# Toy model: strip (...) blocks.  Same structure as <think>...</think>
# but single-char delimiters keep the alphabet small → enumerable.
TINY = set('ab()')

def strip_parens(alphabet):
    """Strip (…) blocks — toy model of strip_anywhere with 1-char delimiters."""
    fst = FST()
    fst.add_start('copy'); fst.add_stop('copy')
    for c in alphabet:
        fst.add_arc('copy', c, c, 'copy')
    fst.add_arc('copy', '(', EPSILON, 'inside')
    for c in alphabet:
        fst.add_arc('inside', c, EPSILON, 'inside')
    fst.add_arc('inside', ')', EPSILON, 'copy')
    return fst

T_toy = strip_parens(TINY)
display(visualize_automaton(T_toy))

### Precover DFA

The precover DFA for a target *y* accepts exactly the source strings whose
output starts with *y*.  Colored states:
**green** = universal (quotient), **magenta** = non-universal accepting (remainder).

In [ ]:
from transduction.viz import display_table

for target_str in ['', 'a', 'ab']:
    target = tuple(target_str) if target_str else ()
    r = Precover(T_toy, target)
    label = target_str or 'ε'
    print(f'Precover DFA for target "{label}": {len(r.det.materialize().states)} states')
    display(r)       # colored precover DFA
    r.show_decomposition()   # Q and R side by side

Every accepting state is **green** (universal) — no magenta remainder states.
This is because the `copy` state always allows continuing with any symbol,
so every source prefix that reaches a final state can also be extended.
Result: **R is always ∅** for this FST.

### `decompose_next`: per-symbol Q languages

`decompose_next()` computes Q(*y* · *c*) for each possible next output
symbol *c*.  This is the key operation for constrained decoding — it tells
you which output symbols are valid continuations and what source strings
support them.

In [ ]:
for target_str in ['', 'a']:
    target = tuple(target_str) if target_str else ()
    r = Precover(T_toy, target)
    label = target_str or 'ε'
    print(f'decompose_next for target "{label}":')
    for y, child in sorted(r.decompose_next().items()):
        Q_strs = sorted([t2s(x) for x in child.quotient.language(7)],
                        key=lambda s: (len(s), s))
        marker = '  ' if y in '()' else '→ '
        print(f'  {marker}"{label}{y}": Q has {len(Q_strs)} strings (≤7): {Q_strs[:6]}{"…" if len(Q_strs) > 6 else ""}')
    print()

The pattern is clear: Q(*y* · *c*) contains the "obvious" source prefix
(just *y* · *c* itself) plus every variant with `(…)` blocks inserted at
any position.  These `(…)` blocks are the hidden reasoning traces —
decomposition tells you exactly which source strings are consistent with
the observed output prefix.

In constrained decoding, this is used to compute
**∑ₓ P(x) · 1[x ∈ Q(y·c)]** — the total probability mass on source
strings that could produce each next output symbol *c*.

---
## Discussion: partial functions

**Is the beginning-only version a partial function?**

Not as built — it's a **total relation** (every input has at least one
output).  When the tag is at the beginning, we get two outputs (stripped
and unstripped), so it's not even functional.

To make it a **partial function** (functional + possibly undefined), we
could build a *deterministic* version that:
- strips a leading `<think>…</think>` if present → exactly one output,
- passes through unchanged if the string doesn't start with `<think>` → one output,
- **rejects** (no output) if `<think>` appears later in the string.

The third case makes the function *partial*: the FST has no accepting
path for inputs that violate the "beginning or not at all" constraint.

However, the version above is more useful in practice: it's total (always
produces output) and simply ignores non-initial tags.  Whether you want
partial-function semantics depends on whether you want to *reject* or
*tolerate* misplaced tags.

### Token-level simplification

In practice with LLMs, `<think>` and `</think>` are usually single tokens
in the vocabulary.  Working at the **token level** instead of the character
level makes the FST trivial — no multi-character matching needed:

```python
fst = FST()
fst.add_start(0); fst.add_stop(0)
for tok in vocab:
    fst.add_arc(0, tok, tok, 0)           # copy
fst.add_arc(0, '<think>', EPSILON, 1)     # suppress open
for tok in vocab:
    fst.add_arc(1, tok, EPSILON, 1)       # suppress content
fst.add_arc(1, '</think>', EPSILON, 0)    # suppress close
```